# Kernel cuantico para QSVM

Construye el kernel de fidelidad ``K(x_i, x_j) = P(00...0)`` del circuito
``U(x_j)^dagger U(x_i)``, con el feature map elegido en `FEATURE_MAP`
(ZZ, Pauli Z+YY o Ry-CX-Rx) definido en pytket y ejecutado via guppy.

Flujo: cargar datos -> inspeccionar circuitos (sin shots) -> enviar y guardar
K_train (para `SVC.fit`) -> enviar y guardar K_test (para `SVC.predict`).
Toda la logica vive en `funciones_nexus.py`; aqui solo quedan los parametros
y las llamadas.

La construccion esta apagada por defecto (`RUN_MATRIX = False`): revisa
circuitos y costo antes de encenderla.

## 1. Configuracion y datos

Carga el dataset escalado y lo separa en train/test segun `_PartInd_`.

In [ ]:
import pandas as pd
from IPython.display import display
from pytket.circuit.display import render_circuit_jupyter

# Los valores del kernel suelen ser pequenos; se muestran con 6 decimales
# fijos (sin notacion cientifica) para poder distinguirlos.
pd.set_option("display.float_format", "{:.6f}".format)

from funciones_nexus import (
    cargar_datos_kernel_muestreo,
    obtener_feature_map,
    seleccionar_par_kernel,
    iniciar_matriz_kernel,
    iniciar_matriz_kernel_test,
    consultar_matriz_nexus,
    guardar_kernel_qsvm,
    FEATURE_MAPS,
    NIVELES_MUESTREO,
    MATRIX_BACKEND_OPTIONS,
)

# Feature map del kernel: elige una de las 3 opciones de FEATURE_MAPS.
#   "zz"       -> ZZFeatureMap (fases Z + interacciones ZZ)
#   "zyy"      -> Pauli Z+YY explicito (entrelazamiento lineal)
#   "ry_cx_rx" -> Ry -> cadena CX -> Rx
# El mismo FEATURE_MAP gobierna K_train y K_test (deben coincidir).
FEATURE_MAP = "zyy"

PROJECT_NAME = "prueba_migracion"                        # Proyecto de Nexus (se crea si no existe)
KERNEL_DATA_PATH = "data/processed/dataset_v1.xlsx"      # Excel v1 con la hoja "muestreos"

# Nivel de muestreo jerarquico, independiente por particion (ver hoja
# "muestreos" en dataset_v1.xlsx). Notacion del usuario -> NIVELES_MUESTREO:
#   "3"     -> solo la submuestra mas chica  (train=16, test=8)
#   "3+2"   -> union de las dos mas chicas   (train=32, test=16)
#   "3+2+1" -> la muestra completa           (train=64, test=32)
TRAIN_MUESTREO = "3+2+1"
TEST_MUESTREO = "3+2+1"

kernel_df, kernel_feature_columns, kernel_train_df, kernel_test_df = cargar_datos_kernel_muestreo(
    KERNEL_DATA_PATH,
    nivel_train=NIVELES_MUESTREO[TRAIN_MUESTREO],
    nivel_test=NIVELES_MUESTREO[TEST_MUESTREO],
)
print("Feature map:", FEATURE_MAP, "| opciones:", list(FEATURE_MAPS))
print(f"Muestreo train: {TRAIN_MUESTREO} | Muestreo test: {TEST_MUESTREO}")
print("Features:", kernel_feature_columns)

Feature map: zyy | opciones: ['zz', 'zyy', 'ry_cx_rx']
Muestreo train: 3 | Muestreo test: 3
Features: ['ph', 'Hardness', 'Solids', 'Chloramines', 'Sulfate', 'Conductivity', 'Organic_carbon', 'Trihalomethanes', 'Turbidity']


### Datos que alimentan a los modelos

La carga y el filtrado viven en `cargar_datos_kernel_muestreo`
(`funciones_nexus.py`): lee la hoja "muestreos" de `dataset_v1.xlsx`
(pipeline v1, con su propio escalado) y separa train/test por
`_PartInd_`, filtrando cada particion por su propio nivel de muestreo
jerarquico (`TRAIN_MUESTREO` / `TEST_MUESTREO`, independientes entre si).
Devuelve **solo las columnas de features** (sin `Potability`,
`_PartInd_` ni `_Muestreo_`). Estas dos tablas son exactamente las que se
codifican en los circuitos: `kernel_train_df` construye K_train y
`kernel_test_df` construye K_test.

Nota: esta hoja no esta alineada fila a fila con
`data/processed/df_escalado.csv` (pipeline del handoff/guppy, escalado
independiente); usar esta funcion implica que el kernel consume las
features del pipeline v1 para las filas seleccionadas.

In [4]:
# Cantidad de registros y vista previa de las tablas que alimentan el kernel.
print(f"Dataset completo : {kernel_df.shape[0]} registros")
print(f"Train (_PartInd_=0): {kernel_train_df.shape[0]} registros | {kernel_train_df.shape[1]} features")
print(f"Test  (_PartInd_=1): {kernel_test_df.shape[0]} registros | {kernel_test_df.shape[1]} features")

print("\nTrain (head) -> alimenta K_train:")
display(kernel_train_df.head())
print("Test (head) -> alimenta K_test:")
display(kernel_test_df.head())

Dataset completo : 96 registros
Train (_PartInd_=0): 16 registros | 9 features
Test  (_PartInd_=1): 8 registros | 9 features

Train (head) -> alimenta K_train:


,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity
0,-0.361979,2.514880,0.413352,-0.229521,0.913676,0.857622,0.694928,0.833280,-0.463888
1,0.841088,0.758451,-0.658195,1.475650,0.397679,0.911400,2.392957,0.582606,-1.555474
2,1.018002,0.570489,-0.697537,0.260433,-0.463996,-0.841309,0.203870,0.842165,0.036332
3,0.670543,-0.686461,-0.059575,0.234273,-0.513103,0.853883,-0.866438,-1.263355,-0.278637
4,1.077910,0.318782,0.874958,-0.679265,0.145257,-1.213481,-1.557325,-0.151884,-0.364706


Test (head) -> alimenta K_test:


,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity
0,1.470856,-2.102325,2.018475,1.091025,-1.429069,0.734271,-1.437603,-0.041013,0.954979
1,-0.030569,-0.748030,0.537460,0.697905,-0.019216,-0.977065,-1.940272,1.718473,-0.542778
2,-1.261818,-0.351513,-0.640716,-0.312252,-0.019216,-0.411240,-1.958454,1.449338,-0.356368
3,-0.017483,0.487514,-1.648467,1.007401,-0.019216,-0.393719,-0.876470,0.573384,1.034145
4,-0.030569,-1.347883,-0.246199,2.358087,1.110126,0.442760,0.382948,-0.317439,1.692371


## 2. Inspeccion del feature map U(x)

No consume shots.

In [5]:
PREVIEW_ROW = 0     # Cambia esta fila para inspeccionar otro U(x), sin ejecutar shots

preview_x = kernel_train_df.iloc[PREVIEW_ROW].to_numpy(dtype=float)
feature_map_fn = obtener_feature_map(FEATURE_MAP)
feature_map_preview = feature_map_fn(preview_x)
print(f"Feature map '{FEATURE_MAP}' de train[{PREVIEW_ROW}] | qubits: {feature_map_preview.n_qubits} | puertas: {feature_map_preview.n_gates}")
render_circuit_jupyter(feature_map_preview)

Feature map 'zyy' de train[0] | qubits: 9 | puertas: 115


## 3. Seleccion e inspeccion del par

Construye ``U(x_j)^dagger U(x_i)`` con barreras para revisarlo antes de ejecutar.

In [6]:
KERNEL_ROW_I = 0    # Filas de train que forman el par
KERNEL_ROW_J = 1

kernel_x_i, kernel_x_j, kernel_preview_circuit = seleccionar_par_kernel(
    kernel_train_df, KERNEL_ROW_I, KERNEL_ROW_J, feature_map=FEATURE_MAP
)
render_circuit_jupyter(kernel_preview_circuit)

Kernel seleccionado: train[0] vs train[1]
Qubits: 9
Puertas: 230
Profundidad: 116


## 4. Enviar la matriz K_train

`K_train = K(X_train, X_train)` (cuadrada, para `SVC.fit`). Se ejecuta el
triangulo superior y se refleja por simetria; para `m` filas de train se
requieren `m(m-1)/2` circuitos (mas la diagonal si `MATRIX_EXECUTE_DIAGONAL`).

Backends: Selene local o Nexus (Selene, H1/H2 via compile job, Helios). En
local el resultado llega aqui mismo; en Nexus se envia el job y se sigue en
el paso 5. **Tras enviar a Nexus no reejecutes esta celda.**

In [ ]:
MATRIX_ROWS = [0, 1, 2, 3]                   # Filas de train (definen K_train y las columnas de K_test)
MATRIX_BACKEND = "nexus_selene_statevector"               # Ver MATRIX_BACKEND_OPTIONS
RUN_MATRIX = True                            # Interruptor de seguridad
MATRIX_SHOTS = 1000
MATRIX_SEED = 42
MATRIX_EXECUTE_DIAGONAL = True               # False fija K(i,i)=1 sin ejecutar
SAVE_MATRIX_RUN = True

matrix_state_train, matrix_result_train = iniciar_matriz_kernel(
    kernel_train_df, MATRIX_ROWS, MATRIX_BACKEND, RUN_MATRIX,
    n_shots=MATRIX_SHOTS, seed=MATRIX_SEED,
    ejecutar_diagonal=MATRIX_EXECUTE_DIAGONAL,
    guardar=SAVE_MATRIX_RUN, project_name=PROJECT_NAME,
    feature_map=FEATURE_MAP,
)

if matrix_result_train is not None:
    display(pd.DataFrame(matrix_result_train["kernel_matrix"], index=MATRIX_ROWS, columns=MATRIX_ROWS))
    display(matrix_result_train["run_summary"])

Already logged in. Tokens are valid.


Preparando hugr: 100%|██████████| 10/10 [00:08<00:00,  1.11circuito/s]


Job de ejecucion HUGR enviado.
Execute Job ID: cce62e06-759d-41ac-9f99-3f913b464e1a
Backend: nexus_selene_statevector
Formato: hugr
Programas: 10
Consulta el avance con la celda de consulta; no reenvies esta celda.


## 5. Consultar y guardar K_train

En Nexus, reejecuta **solo esta celda** hasta que el job llegue a COMPLETED
(H1/H2 encadena compile -> execute automaticamente). En local no hay nada que
consultar (`matrix_state_train` es None) y se guarda directo la matriz del
paso 4. Persiste la matriz de Gram cuadrada como `kernel_qsvm_<run_id>.csv`
(lista para `SVC(kernel="precomputed")`) + metadatos.

In [5]:
matrix_state_train, matrix_train_remoto = consultar_matriz_nexus(matrix_state_train, guardar=SAVE_MATRIX_RUN)

# Toma la matriz remota si ya llego; si no, la local del paso 4.
if matrix_train_remoto is not None:
    K_train_final = matrix_train_remoto
    fuente_train = f"nexus_{MATRIX_BACKEND}_train"
    id_train = matrix_state_train["job_ref"].id
elif matrix_result_train is not None:
    K_train_final = matrix_result_train
    fuente_train = "local_statevector_train"
    id_train = None
else:
    K_train_final = None
    print("K_train aun no disponible (job remoto en curso o sin construir).")

if K_train_final is not None:
    ruta_train, ruta_train_meta = guardar_kernel_qsvm(K_train_final, source=fuente_train, job_id=id_train)
    print("K_train guardada en:", ruta_train)
    print("Metadatos en:", ruta_train_meta)
    display(pd.read_csv(ruta_train, sep=";", index_col=0))

Execute Job ID: cce62e06-759d-41ac-9f99-3f913b464e1a
Execute status: JobStatusEnum.COMPLETED
Execute message: The job is completed.
Matriz kernel reconstruida.
Run de matriz guardado en: data\runs\kernel_matrix_run_cce62e06-759d-41ac-9f99-3f913b464e1a.csv
K_train guardada en: data\runs\kernel_qsvm_cce62e06-759d-41ac-9f99-3f913b464e1a.csv
Metadatos en: data\runs\kernel_qsvm_cce62e06-759d-41ac-9f99-3f913b464e1a_meta.csv


,0,1,2,3
0,1.000000,0.000000,0.000000,0.000000
1,0.000000,1.000000,0.000000,0.100000
2,0.000000,0.000000,1.000000,0.000000
3,0.000000,0.100000,0.000000,1.000000


## 6. Enviar la matriz K_test

`K_test = K(X_test, X_train)` (rectangular n_test x m, para `SVC.predict`).
Reutiliza los mismos parametros del paso 4 (backend, shots, feature map) y las
mismas `MATRIX_ROWS` como columnas. Internamente apila `[test, train]`,
construye la conjunta y recorta el bloque test x train.

Es un **job independiente** del de K_train: en Nexus puedes lanzar este envio
sin esperar a que termine el de train, y consultar cada uno por separado.

In [6]:
TEST_ROWS = [0, 1, 2, 3]                     # Filas de test (filas de K_test); columnas = MATRIX_ROWS

matrix_state_test, matrix_result_test = iniciar_matriz_kernel_test(
    kernel_train_df, MATRIX_ROWS, kernel_test_df, MATRIX_BACKEND, RUN_MATRIX,
    test_rows=TEST_ROWS, n_shots=MATRIX_SHOTS, seed=MATRIX_SEED,
    guardar=SAVE_MATRIX_RUN, project_name=PROJECT_NAME,
    feature_map=FEATURE_MAP,
)

if matrix_result_test is not None:
    display(pd.DataFrame(matrix_result_test["kernel_matrix"], index=TEST_ROWS, columns=MATRIX_ROWS))
    display(matrix_result_test["run_summary"])

Already logged in. Tokens are valid.


Preparando hugr: 100%|██████████| 28/28 [00:24<00:00,  1.16circuito/s]


Job de ejecucion HUGR enviado.
Execute Job ID: fa7ba5ce-ec90-43fb-8cd5-c1508401a115
Backend: nexus_selene_statevector
Formato: hugr
Programas: 28
Consulta el avance con la celda de consulta; no reenvies esta celda.
Nexus construira la conjunta (8, 8) ; al consultar se recorta K_test (4, 4)


## 7. Consultar y guardar K_test

Igual que el paso 5 pero para el job de test. Persiste la matriz rectangular
como `kernel_qsvm_test_<run_id>.csv` (filas = test, columnas = train). Con
K_train (paso 5) y K_test (aqui) ya tienes ambos artefactos para el QSVM:
`SVC(kernel="precomputed").fit(K_train, y_train).predict(K_test)`.

In [7]:
matrix_state_test, matrix_test_remoto = consultar_matriz_nexus(matrix_state_test, guardar=SAVE_MATRIX_RUN)

if matrix_test_remoto is not None:
    K_test_final = matrix_test_remoto
    fuente_test = f"nexus_{MATRIX_BACKEND}_test"
    id_test = matrix_state_test["job_ref"].id
elif matrix_result_test is not None:
    K_test_final = matrix_result_test
    fuente_test = "local_statevector_test"
    id_test = None
else:
    K_test_final = None
    print("K_test aun no disponible (job remoto en curso o sin construir).")

if K_test_final is not None:
    ruta_test, ruta_test_meta = guardar_kernel_qsvm(K_test_final, source=fuente_test, job_id=id_test)
    print("K_test guardada en:", ruta_test)
    print("Metadatos en:", ruta_test_meta)
    display(pd.read_csv(ruta_test, sep=";", index_col=0))

Execute Job ID: fa7ba5ce-ec90-43fb-8cd5-c1508401a115
Execute status: JobStatusEnum.COMPLETED
Execute message: The job is completed.
Matriz K_test reconstruida (bloque test x train recortado).
Run de matriz guardado en: data\runs\kernel_matrix_run_fa7ba5ce-ec90-43fb-8cd5-c1508401a115.csv
K_test guardada en: data\runs\kernel_qsvm_fa7ba5ce-ec90-43fb-8cd5-c1508401a115.csv
Metadatos en: data\runs\kernel_qsvm_fa7ba5ce-ec90-43fb-8cd5-c1508401a115_meta.csv


,0,1,2,3
0,0.000000,0.000000,0.000000,0.000000
1,0.000000,0.000000,0.000000,0.000000
2,0.000000,0.000000,0.000000,0.000000
3,0.000000,0.000000,0.000000,0.000000
